# **03 - XGBoost**

This notebook trains the snapshot-based XGBoost model on `snapshot_features` from `02_features_snapshot.ipynb`. Coverage-scheme variables were dropped as mechanically unrelated to a single blocking matchup, while rusher count and a derived blitz flag were added to capture how the number of rushers affects blocking assignments. Class imbalance was handled with `scale_pos_weight`, tuned via a sensitivity sweep and set to bias toward recall, since missing a real blocking breakdown is more costly than over-flagging one. A grid search over tree depth, learning rate, and number of estimators was run with that weight held fixed, and the tuned model was evaluated with AUC, log-loss, and a full classification report. SHAP values were computed to interpret which features drive individual predictions, and the model and test-set predictions were saved for the writeup and the comparison against the LSTM in `04_lstm.ipynb`.

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb, matplotlib.pyplot as plt, shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, classification_report

snapshot_features = pd.read_parquet("../processed/snapshot_features.parquet")
print(f"Loaded {len(snapshot_features):,} rows")

In [ ]:
categorical_cols = [
    "blocker_position", "rusher_position", "offenseFormation"
]
numeric_cols = [
    "dist_snap", "dist_1s", "dist_final",
    "closing_speed_1s", "closing_speed_final",
    "orientation_diff_snap", "orientation_diff_final",
    "weight_diff", "down", "yardsToGo", "rusher_count"
]
flag_cols = ["assisted_block", "pff_playAction", "blitz_flag"]

feature_df = snapshot_features[categorical_cols + numeric_cols + flag_cols].copy()
target = snapshot_features["blocker_lost"]
feature_df[flag_cols] = feature_df[flag_cols].astype(int)
feature_df = pd.get_dummies(feature_df, columns = categorical_cols, drop_first = False)

print(f"Feature matrix shape: {feature_df.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    feature_df, target, test_size = 0.2, random_state = 42, stratify = target
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}, Test positive rate: {y_test.mean():.3f}")

In [ ]:
weight_candidates = [5.0, 5.5, 6.0, 6.5, 7.0, 7.5]
results = []
for w in weight_candidates:
    m = xgb.XGBClassifier(
        n_estimators = 300,
        max_depth = 4,
        learning_rate = 0.05,
        scale_pos_weight = w,
        eval_metric = "logloss",
        random_state = 42,
    )
    m.fit(X_train, y_train, verbose = False)

    proba = m.predict_proba(X_test)[:, 1]
    pred = m.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    ll = log_loss(y_test, proba)
    report = classification_report(y_test, pred, output_dict = True)

    results.append({
        "scale_pos_weight": w,
        "AUC": auc,
        "log_loss": ll,
        "precision_1": report["1"]["precision"],
        "recall_1": report["1"]["recall"],
        "f1_1": report["1"]["f1-score"],
    })

sensitivity_df = pd.DataFrame(results)
sensitivity_df

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300, 400],
}

base_model = xgb.XGBClassifier(
    scale_pos_weight=6.5,
    eval_metric = "logloss",
    random_state = 42,
)

grid_search = GridSearchCV(
    base_model,
    param_grid,
    scoring="f1",
    cv=3,
    n_jobs=1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")

In [ ]:
best_model = grid_search.best_estimator_

y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
ll = log_loss(y_test, y_pred_proba)

print(f"Test AUC: {auc:.4f}")
print(f"Test log-loss: {ll:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import shap

explainer = shap.TreeExplainer(best_model, feature_perturbation = "tree_path_dependent")
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, show=False)
fig = plt.gcf()
fig.savefig("../figures/xgboost_shap_summary.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
test_indices = X_test.index
results = snapshot_features.loc[test_indices, ["gameId", "playId", "nflId", "rusher_nflId"]].copy()
results["blocker_lost_actual"] = y_test.values
results["blocker_lost_predicted_proba"] = y_pred_proba
results["blocker_win_probability"] = 1 - results["blocker_lost_predicted_proba"]
best_model.save_model("xgboost_snapshot.json")
results.to_parquet("xgboost_predictions.parquet", index=False)
print("Model and predictions saved.")